<a href="https://colab.research.google.com/github/prbchinmayi/Deforestration_detection_ResNet50/blob/main/eurosat_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#check 1
import torch
print(torch.cuda.is_available())


#check 2
print("python version:",torch.__version__, torch.device("cuda" if torch.cuda.is_available() else "cpu"))
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

True
python version: 2.11.0+cu128 cuda
Tesla T4


In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

In [43]:
from torchvision.datasets import EuroSAT

transform= transforms.Compose([transforms.Resize((224,224)), # ResNet-50 expects 224x224 px
                               transforms.ToTensor(),
                               transforms.Normalize(
                                   mean=[0.485, 0.456, 0.406],
                                   std=[0.229, 0.224, 0.225]
                               )
                             ])

dataset=EuroSAT(root="./data", transform=transform,download=True)
classes=dataset.classes
size=len(dataset)

#dataset info
print(f"no of classes:{len(classes)}")
print(f"Classes:")
for cl in (classes):
  print(" ", cl)
print(f"no of images:{size}")


no of classes:10
Classes:
  AnnualCrop
  Forest
  HerbaceousVegetation
  Highway
  Industrial
  Pasture
  PermanentCrop
  Residential
  River
  SeaLake
no of images:27000


In [59]:
#splitting dataset
train_size=int(0.70*size)
val_size=int(0.15*size)
test_size=int(0.15*size)

train_set, val_set, test_set= random_split(dataset,[train_size, val_size, test_size], generator= torch.Generator().manual_seed(42)) #same split each time
print(f"training: {len(train_set)}")
print(f"validation: {len(val_set)}")
print(f"testing: {len(test_set)}")

#dataloaders to divide images in batches for model to process
batch_size=32

train_loader= DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader= DataLoader(val_set, batch_size=batch_size, shuffle=True)
test_loader= DataLoader(test_set, batch_size=batch_size, shuffle=True)

training: 18900
validation: 4050
testing: 4050


In [60]:
#pretrained ResNet50
#aldready learned visual features from ImageNet so we just transfer this learning to sat images

weigths=models.ResNet50_Weights.DEFAULT
model=models.resnet50(weights=weigths)

#only train the final layer classifier
#lock feature extraction layers
for para in model.parameters():
  para.requires_grad= False

#check for classes division in ResNet50
print(model.fc)

Linear(in_features=2048, out_features=1000, bias=True)


In [61]:
#replace 1000 ImageNet classes with 10 EuroSAT classes
no_classes= 10
model.fc= nn.Linear(model.fc.in_features, no_classes)
print(model.fc)

Linear(in_features=2048, out_features=10, bias=True)


In [62]:
#move model to gpu
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model= model.to(device)

cuda


In [63]:
print(torch.is_grad_enabled())
print(model.fc.weight.requires_grad)

model.fc.bias.requires_grad = True

True
True


In [64]:
#for loss function
criterion= nn.CrossEntropyLoss()

optimiser=optim.Adam(model.fc.parameters(), lr=0.001)

no_epochs=5
for epoch in range(no_epochs):
  #training
  model.train()
  runningloss=0.0
  correct=0
  total=0

  for imgs, labels in train_loader:
    imgs=imgs.to(device)
    labels=labels.to(device)

    optimiser.zero_grad()
    outputs=model(imgs)
    loss= criterion(outputs,labels)
    loss.backward()
    optimiser.step()

    runningloss= runningloss+loss.item()

    _, predicted= torch.max(outputs,1)
    total= total+labels.size(0)
    correct= correct+(predicted==labels).sum().item()

  train_acc=(correct/total)*100

  #validation
  model.eval()

  val_correct=0
  val_total=0

  with torch.no_grad():
    for imgs, labels in val_loader:
      imgs=imgs.to(device)
      labels=labels.to(device)

      outputs=model(imgs)

      _, predicted= torch.max(outputs,1)
      val_total= val_total+labels.size(0)
      val_correct= val_correct+(predicted==labels).sum().item()
  val_acc=(val_correct/val_total)*100

  print(f"epoch[{epoch+1}/{no_epochs}]")
  print(f"train accuracy: {train_acc:.2f}")
  print(f"validation accuracy: {val_acc:.2f}")



epoch[1/5]
train accuracy: 81.63
validation accuracy: 89.88
epoch[2/5]
train accuracy: 89.58
validation accuracy: 92.17
epoch[3/5]
train accuracy: 91.34
validation accuracy: 93.09
epoch[4/5]
train accuracy: 91.89
validation accuracy: 93.38
epoch[5/5]
train accuracy: 92.82
validation accuracy: 93.70


In [65]:
model.eval()
test_correct=0
test_total=0

with torch.no_grad():
  for imgs, labels in test_loader:
    imgs=imgs.to(device)
    labels=labels.to(device)

    outputs=model(imgs)

    _, predicted= torch.max(outputs,1)
    test_total= test_total+labels.size(0)
    test_correct= test_correct+(predicted==labels).sum().item()
test_acc=(test_correct/test_total)*100
print(f"testing accuracy: {test_acc:.2f}")

testing accuracy: 93.70
